[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C14/E14-2_Pieces_8x8.ipynb)

# Contrôle de rondelles par caméra : trouver le type de défaut

## Le contexte

Une caméra de contrôle photographie chaque rondelle usinée en sortie de ligne. L'image est très petite, 8 sur 8 pixels, et la pièce y occupe une surface de quelques pixels seulement. Une rondelle peut avoir un diamètre hors cote, un perçage décentré, une forme ovale, un perçage hors cote ou une ébréchure sur la rive. Le service qualité ne veut plus seulement savoir si la pièce est bonne : il veut le type de défaut, pour corriger la bonne opération en amont. Un échantillon de 4 000 rondelles a été photographié et classé par un contrôleur.

La grandeur à prédire est une catégorie parmi six : c'est un problème de classification.

Défi : prédire `type_defaut` et obtenir un F-score macro supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ est le réseau de neurones de scikit-learn du chapitre précédent, aux réglages par défaut (une couche de 100 neurones) ; il atteint un F-score macro de 0,49. À vous de faire mieux avec un réseau dense de Keras, dont vous réglerez la largeur, la régularisation et le nombre d'époques sur une validation. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Pieces_8x8.xlsx`, dans le dossier `donnees` du dépôt : 4 000 images, 66 colonnes. Une ligne est l'image 8 × 8 d'une pièce vue par la caméra de contrôle.

| Colonne | Signification | Unité |
|---|---|---|
| `p00` à `p63` | pixels de l'image, lus ligne par ligne : `p00` en haut à gauche, `p07` en haut à droite, `p63` en bas à droite | niveau de gris, 0 noir et 255 blanc |
| `conforme` | 1 si la pièce respecte toutes les tolérances, seconde cible du jeu, écartée ici : elle se déduit du type de défaut | 0 ou 1 |
| `type_defaut` | cible : `aucun`, `diametre`, `decentrage`, `ovalisation`, `percage` ou `ebrechure` | catégorie |

Origine : jeu simulé, rendu géométrique d'une rondelle claire sur fond sombre, avec des défauts de dimension et de forme, une position qui varie dans le champ, un éclairage variable et le bruit du capteur. Il enseigne une méthode, et aucune conclusion de métrologie n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et la répartition des classes avant toute modélisation : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Pieces_8x8.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Pieces_8x8.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "type_defaut"
pixels = [f"p{k:02d}" for k in range(64)]
print("dimensions :", df.shape)
print("cases vides :", int(df.isna().sum().sum()))
print(df[cible].value_counts().to_dict())
df[pixels[:5] + [cible]].head(3)

Lecture du résultat : 4 000 images, 66 colonnes, aucune case vide. La moitié des pièces n'a aucun défaut ; chacun des cinq défauts en représente un dixième. Un modèle qui répondrait toujours « aucun » aurait raison une fois sur deux, mais ne trouverait aucun défaut : le F-score macro, qui moyenne le F-score de chaque classe, le punit, alors que l'exactitude le laisserait à 50 %.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Pieces_8x8.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Reproductibilité

Un réseau de neurones part de poids tirés au hasard et lit ses exemples dans un ordre tiré au hasard. La cellule suivante fixe toutes les graines, demande des opérations de calcul déterministes, impose un seul fil de calcul et un jeu d'instructions donné. Avec elle, deux exécutions du notebook sur le même ordinateur donnent les mêmes poids et le même score. Chaque cellule qui construit un réseau rappelle la graine dans sa première ligne : on peut la rejouer seule sans changer le résultat. Sur un autre ordinateur, ou sur un processeur qui mêle deux types de cœurs, les arrondis peuvent différer et le score changer à la troisième décimale.

In [ ]:
# Réseaux de neurones : mêmes résultats à chaque exécution.
# À placer après la cellule de lecture (qui définit GRAINE),
# avant tout usage de TensorFlow.
import os

# Lues à l'import de TensorFlow : journal réduit, oneDNN limité
# au produit matriciel, jeu d'instructions fixé.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["ONEDNN_MAX_CPU_ISA"] = "AVX2"

import numpy as np
import tensorflow as tf
import keras

# Graines, opérations déterministes et un seul fil de calcul.
keras.utils.set_random_seed(GRAINE)
tf.config.experimental.enable_op_determinism()
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)

print("TensorFlow", tf.__version__, "| Keras", keras.__version__,
      "| NumPy", np.__version__)

### Étape 2 : mettre le test de côté

Un quart des images est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la cible. Le tirage est stratifié sur le type de défaut, pour que chaque classe soit présente en proportion identique des deux côtés. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent. Dans l'apprentissage restant, un cinquième est réservé à la validation : il servira à régler le réseau et à décider quand arrêter l'entraînement, jamais le test. La conformité, seconde cible du jeu, ne sert pas d'entrée.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[pixels]
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=GRAINE, stratify=y)
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=GRAINE, stratify=y_train)
print("apprentissage :", len(X_fit), "images")
print("validation :", len(X_val), "images")
print("test :", len(X_test), "images")

### Étape 3 : regarder les images de l'apprentissage

La figure montre cinq images tirées au hasard de l'apprentissage pour chaque type de défaut. Le tirage ne touche que `X_train`.

In [ ]:
classes = ["aucun", "diametre", "decentrage", "ovalisation", "percage", "ebrechure"]
rng = np.random.default_rng(GRAINE)
fig, axes = plt.subplots(len(classes), 5, figsize=(5.2, 6.0))
for i, classe in enumerate(classes):
    for j, position in enumerate(rng.choice(X_train.index[y_train == classe], 5, replace=False)):
        axes[i, j].imshow(X_train.loc[position].to_numpy().reshape(8, 8), cmap="gray", vmin=0, vmax=255)
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])
    axes[i, 0].set_ylabel(classe, rotation=0, ha="right", va="center", fontsize=9)
plt.tight_layout()
plt.show()
contraste = X_train.max(axis=1) - X_train.min(axis=1)
print(f"niveau de gris le plus clair moins le plus sombre : {contraste.min():.0f} à {contraste.max():.0f} (médiane {contraste.median():.0f})")

Lecture du résultat : sur 64 pixels, l'œil distingue une rondelle claire avec son perçage, parfois décalée dans le champ ; les défauts de dimension (un diamètre ou un perçage un peu trop grand ou trop petit) et de forme (un perçage excentré, une ovalisation légère, une ébréchure) ne se voient pas à coup sûr sur une image aussi pauvre. La tâche est donc difficile : la différence entre deux images de classes voisines tient parfois à un demi-pixel. Le contraste de chaque image varie, de 129 à 255 niveaux de gris, puisque l'éclairage change d'une image à l'autre : la mise à l'échelle des entrées comptera.

### Étape 4 : construire le modèle de départ

Le modèle de départ est le réseau de neurones de scikit-learn du chapitre précédent, aux réglages par défaut : une couche cachée de 100 neurones, 200 époques au plus. Un pipeline ramène d'abord chaque pixel à une moyenne nulle et à un écart type de 1, puis ajuste le réseau sur `X_train` seul.

In [ ]:
# Votre code :

### Étape 5 : mesurer sur la base de test

Le modèle prédit les images qu'il n'a jamais vues. Deux indicateurs sont lus : le F-score macro, moyenne des F-scores de chaque classe, et l'exactitude, part des images bien classées. Un témoin sans apprentissage, qui répond toujours la classe la plus fréquente, donne le point de comparaison le plus bas.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

y_pred = modele.predict(X_test)
score_depart = f1_score(y_test, y_pred, average="macro")
temoin = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"F-score macro : {score_depart:.3f}")
print(f"exactitude : {accuracy_score(y_test, y_pred):.3f}")
print(f"F-score macro du témoin (toujours la classe la plus fréquente) : {f1_score(y_test, temoin.predict(X_test), average='macro'):.3f}")
print(f"exactitude du témoin : {accuracy_score(y_test, temoin.predict(X_test)):.3f}")
print(f"époques effectuées : {modele[-1].n_iter_} sur {modele[-1].max_iter} permises")

Lecture du résultat : le F-score macro est de 0,49 et l'exactitude de 0,61. Le témoin, qui répond toujours « aucun », a une exactitude de 0,50 et un F-score macro de 0,11 : l'exactitude du réseau ne dépasse celle du témoin que de 0,11, parce que la moitié des pièces n'ont aucun défaut, et le F-score macro montre que beaucoup de défauts échappent au modèle. C'est le F-score macro, mesuré sur des images que le modèle n'a jamais vues, que l'on a le droit d'annoncer.

## Pour aller plus loin : un réseau dense de Keras

Le modèle de départ de l'étape 4 reste le score de référence du défi. Les cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première prépare les entrées pour Keras : les pixels sont centrés et réduits avec la moyenne et l'écart type de la partie d'apprentissage seule, et les classes deviennent des entiers. La deuxième construit des réseaux denses de largeurs et de régularisations différentes et les compare sur la validation, avec un arrêt précoce qui suit l'erreur de validation et rend les poids de la meilleure époque. La troisième trace l'erreur du réseau retenu et celle d'un réseau sans régularisation. La quatrième lit, sur la validation, les types de défaut que le réseau confond. La dernière mesure le réseau retenu une seule fois sur le test. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

In [ ]:
# Votre code :

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
for ax, nom, titre in [(axes[0], "128-64", "sans régularisation"), (axes[1], "128-64, dropout 0,2", "avec dropout 0,2")]:
    h = histoires[nom].history
    ax.plot(range(1, len(h["loss"]) + 1), h["loss"], color="0.55", label="apprentissage")
    ax.plot(range(1, len(h["val_loss"]) + 1), h["val_loss"], color="black", linestyle="--", label="validation")
    ax.axvline(int(np.argmin(h["val_loss"])) + 1, color="0.3", linewidth=0.8, linestyle=":")
    ax.set(title=titre, xlabel="époque")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("entropie croisée")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()
ecarts = {nom: histoires[nom].history["val_loss"][-1] - histoires[nom].history["loss"][-1] for nom in ("128-64", "128-64, dropout 0,2")}
for nom, ecart_nom in ecarts.items():
    h = histoires[nom].history
    print(f"{nom} : erreur de validation {h['val_loss'][-1]:.2f}, d'apprentissage {h['loss'][-1]:.2f} à la dernière époque, écart {ecart_nom:.2f}")

In [ ]:
# Votre code :

In [ ]:
# Votre code :

Lecture du résultat : sur la validation, le réseau de scikit-learn par défaut atteint un F-score macro de 0,45. Les cinq réseaux de Keras comparés vont de 0,51 pour une seule couche de 64 neurones à 0,66 : la validation retient « 256-128-64, dropout 0,3 ». Sans régularisation, le réseau 128-64 apprend l'apprentissage par cœur : à la dernière époque, l'erreur de validation dépasse celle d'apprentissage de 0,55 ; le dropout, qui éteint au hasard une part des neurones à chaque pas, ramène cet écart à 0,30. Le réseau retenu, mesuré une seule fois sur le test, atteint un F-score macro de 0,68, soit 0,19 de plus que le modèle de départ. Reste ce que la validation montre par classe : le réseau retrouve 95 % des images de la classe la mieux lue, mais seulement 33 % des ovalisations et 12 % des ébréchures, qu'il prend le plus souvent pour des pièces sans défaut : ces deux défauts de forme déplacent la rive de la pièce de moins d'un pixel, et un réseau dense, qui lit chaque pixel isolément, ne voit pas la forme.

## Le score de référence

Le score publié est le F-score macro du modèle de départ de l'étape 4 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = f1_score(y_test, modele.predict(X_test), average="macro")
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE f1_macro = %.2f" % score)

## Synthèse

1. Une image 8 × 8 devient un vecteur de 64 entrées : le réseau dense ne connaît pas le voisinage des pixels, il apprend un poids par pixel et par neurone.
2. Aux réglages par défaut, le réseau de scikit-learn atteint un F-score macro de 0,49 sur le test, et son exactitude de 0,61, soutenue par la moitié de pièces sans défaut, cache les défauts qui échappent : le F-score macro est la mesure du défi.
3. La largeur, le dropout et le nombre d'époques se règlent sur une validation prise dans l'apprentissage ; l'arrêt précoce rend les poids de la meilleure époque. Ici, le réseau « 256-128-64, dropout 0,3 » est retenu, à 0,66 sur la validation.
4. Sur la validation, les ovalisations et les ébréchures sont presque toutes prises pour des pièces sans défaut : lire des motifs locaux plutôt que des pixels isolés est ce qu'apporte le réseau convolutif du chapitre suivant.
5. La graine est rappelée dans la cellule qui construit le réseau, et le test ne sert qu'une fois, au réseau retenu (0,68 ici).